# Collect Wrong Predictions from BERT, mDeBERTa, and Qwen2-7B Models - TrGLUE Only

This notebook evaluates:
- `emrecan/bert-base-turkish-cased-allnli_tr`
- `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`
- `Qwen/Qwen2-7B-Instruct`

On TrGLUE MNLI (test_matched, test_mismatched) and exports wrong predictions to CSV files.

In [8]:
# Install Required Libraries (run if needed)
!pip install -q datasets transformers torch scikit-learn pandas accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [9]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, pipeline
import torch
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
import json
from collections import Counter

LABELS = ["entailment", "neutral", "contradiction"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Load TrGLUE dataset only
print("Loading TrGLUE MNLI dataset...")
trglue_ds = load_dataset("yilmazzey/sdp2-nli", "trglue_mnli")
print("Dataset loaded.")
print("  splits:", list(trglue_ds.keys()))

Using device: mps
Loading TrGLUE MNLI dataset...
Dataset loaded.
  splits: ['train', 'validation_matched', 'validation_mismatched', 'test_matched', 'test_mismatched']


In [10]:
def evaluate_and_collect_wrongs_bert(model, tokenizer, split_name, output_dir):
    """
    Evaluate BERT model on TrGLUE split and collect wrong predictions.
    Uses batched inference matching base_results/bert-base-turkish-cased-allnli_tr.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    print(f"\nEvaluating on trglue_mnli / {split_name}...")
    
    ds = trglue_ds[split_name]
    premises = ds["premise"]
    hypotheses = ds["hypothesis"]
    labels = np.array(ds["label"])
    
    # Batched inference
    predictions = []
    batch_size = 32
    
    with torch.no_grad():
        for i in tqdm(range(0, len(premises), batch_size), desc="Inference"):
            batch_premises = premises[i:i+batch_size]
            batch_hypotheses = hypotheses[i:i+batch_size]
            
            inputs = tokenizer(
                batch_premises, 
                batch_hypotheses, 
                truncation=True, 
                padding=True, 
                max_length=512,
                return_tensors="pt"
            ).to(DEVICE)
            
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            predictions.extend(preds)
    
    predictions = np.array(predictions)
    
    # Compute metrics
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_per_class = f1_score(labels, predictions, average=None, zero_division=0)
    cm = confusion_matrix(labels, predictions)
    
    print(f"  Accuracy: {accuracy:.4f}, F1 Macro: {f1_macro:.4f}")
    print(f"  F1 per class: {[f'{LABELS[i]}: {f1_per_class[i]:.4f}' for i in range(3)]}")
    print(f"  Confusion matrix:\n{cm}")
    
    # Collect wrong predictions
    wrong_data = []
    for i in range(len(labels)):
        if labels[i] != predictions[i]:
            wrong_data.append({
                "index": i,
                "premise": premises[i],
                "hypothesis": hypotheses[i],
                "true_label": LABELS[labels[i]],
                "predicted_label": LABELS[predictions[i]],
                "split": split_name
            })
    
    # Save to CSV
    if wrong_data:
        df = pd.DataFrame(wrong_data)
        csv_path = Path(output_dir) / f"wrong_trglue_mnli_{split_name}.csv"
        df.to_csv(csv_path, index=False)
        print(f"  Saved {len(wrong_data)} wrong predictions to {csv_path}")
    
    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_per_class": {LABELS[i]: float(f1_per_class[i]) for i in range(3)},
        "confusion_matrix": cm.tolist()
    }

In [ ]:
def evaluate_and_collect_wrongs_mdeberta(model, tokenizer, split_name, output_dir):
    """
    Evaluate mDeBERTa model on TrGLUE split and collect wrong predictions.
    Uses batched inference matching base_results/mDeBERTa-v3-base-mnli-xnli.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    print(f"\nEvaluating on trglue_mnli / {split_name}...")
    
    ds = trglue_ds[split_name]
    
    # Tokenize dataset
    def tokenize_fn(examples):
        return tokenizer(
            examples["premise"],
            examples["hypothesis"],
            truncation=True,
            max_length=256,
        )
    
    # Remove all columns except label
    remove_cols = [c for c in ds.column_names if c != "label"]
    tokenized_ds = ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=remove_cols,
        desc="Tokenize",
    )
    
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    def collate_fn(examples):
        labels = torch.tensor([ex["label"] for ex in examples])
        batch = data_collator([{k: v for k, v in ex.items() if k != "label"} for ex in examples])
        batch["labels"] = labels
        return batch
    
    # Create dataloader
    from torch.utils.data import DataLoader
    loader = DataLoader(tokenized_ds, batch_size=32, collate_fn=collate_fn)
    
    # Inference
    preds_list, labels_list = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            out = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            )
            preds_list.append(out.logits.argmax(-1).cpu().numpy())
            labels_list.append(batch["labels"].numpy())
    
    predictions = np.concatenate(preds_list)
    labels = np.concatenate(labels_list)
    
    # Compute metrics
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_per_class = f1_score(labels, predictions, average=None, zero_division=0)
    cm = confusion_matrix(labels, predictions)
    
    print(f"  Accuracy: {accuracy:.4f}, F1 Macro: {f1_macro:.4f}")
    print(f"  F1 per class: {[f'{LABELS[i]}: {f1_per_class[i]:.4f}' for i in range(3)]}")
    print(f"  Confusion matrix:\n{cm}")
    
    # Collect wrong predictions
    premises = ds["premise"]
    hypotheses = ds["hypothesis"]
    wrong_data = []
    
    for i in range(len(labels)):
        if labels[i] != predictions[i]:
            wrong_data.append({
                "index": i,
                "premise": premises[i],
                "hypothesis": hypotheses[i],
                "true_label": LABELS[labels[i]],
                "predicted_label": LABELS[predictions[i]],
                "split": split_name
            })
    
    # Save to CSV
    if wrong_data:
        df = pd.DataFrame(wrong_data)
        csv_path = Path(output_dir) / f"wrong_trglue_mnli_{split_name}.csv"
        df.to_csv(csv_path, index=False)
        print(f"  Saved {len(wrong_data)} wrong predictions to {csv_path}")
    
    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_per_class": {LABELS[i]: float(f1_per_class[i]) for i in range(3)},
        "confusion_matrix": cm.tolist()
    }

In [12]:
def evaluate_and_collect_wrongs_qwen(generator, tokenizer, split_name, output_dir):
    """
    Evaluate Qwen2-7B model on TrGLUE split and collect wrong predictions.
    Uses prompted inference matching base_results/Qwen2-7B-Instruct.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    print(f"\nEvaluating on trglue_mnli / {split_name}...")
    
    ds = trglue_ds[split_name]
    premises = list(ds["premise"])
    hypotheses = list(ds["hypothesis"])
    labels = np.array(ds["label"])
    n = len(ds)
    
    def nli_prompt(premise, hypothesis):
        return [
            {"role": "system", "content": "You are a helpful assistant for natural language inference. Classify the relationship between premise and hypothesis as entailment, neutral, or contradiction. Respond with exactly one word only: entailment, neutral, or contradiction. No explanation, no other text."},
            {"role": "user", "content": f"Premise: {premise}\nHypothesis: {hypothesis}\nLabel:"}
        ]
    
    def parse_generated_label(gen_text, formatted_prompt):
        continuation = gen_text[len(formatted_prompt):].strip().lower()
        if not continuation:
            return 1  # neutral fallback
        first_word = continuation.split()[0].rstrip('.,!?;:')
        label_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
        return label_map.get(first_word, 1)
    
    # Batched inference
    predictions = []
    batch_size = 16
    all_generations = []
    
    for start in tqdm(range(0, n, batch_size), desc="Inference"):
        end = min(start + batch_size, n)
        batch_premises = premises[start:end]
        batch_hypotheses = hypotheses[start:end]
        batch_prompts = [nli_prompt(p, h) for p, h in zip(batch_premises, batch_hypotheses)]
        
        formatted_prompts = tokenizer.apply_chat_template(batch_prompts, tokenize=False, add_generation_prompt=True)
        
        out = generator(
            formatted_prompts,
            max_new_tokens=5,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            max_length=None,
        )
        
        all_generations.extend(out)
        
        for i, gen in enumerate(out):
            gen_text = gen[0]["generated_text"]
            parsed = parse_generated_label(gen_text, formatted_prompts[i])
            predictions.append(parsed)
    
    predictions = np.array(predictions)
    
    # Compute metrics
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    f1_per_class = f1_score(labels, predictions, average=None, zero_division=0)
    cm = confusion_matrix(labels, predictions)
    
    print(f"  Accuracy: {accuracy:.4f}, F1 Macro: {f1_macro:.4f}")
    print(f"  F1 per class: {[f'{LABELS[i]}: {f1_per_class[i]:.4f}' for i in range(3)]}")
    print(f"  Confusion matrix:\n{cm}")
    
    # Collect wrong predictions
    wrong_data = []
    for i in range(len(labels)):
        if labels[i] != predictions[i]:
            wrong_data.append({
                "index": i,
                "premise": premises[i],
                "hypothesis": hypotheses[i],
                "true_label": LABELS[labels[i]],
                "predicted_label": LABELS[predictions[i]],
                "split": split_name
            })
    
    # Save to CSV
    if wrong_data:
        df = pd.DataFrame(wrong_data)
        csv_path = Path(output_dir) / f"wrong_trglue_mnli_{split_name}.csv"
        df.to_csv(csv_path, index=False)
        print(f"  Saved {len(wrong_data)} wrong predictions to {csv_path}")
    
    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_per_class": {LABELS[i]: float(f1_per_class[i]) for i in range(3)},
        "confusion_matrix": cm.tolist()
    }

## Evaluate BERT-base-turkish-cased-allnli_tr

In [13]:
print("=" * 80)
print("Loading BERT-base-turkish-cased-allnli_tr...")
print("=" * 80)

bert_model_name = "emrecan/bert-base-turkish-cased-allnli_tr"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_model_name, num_labels=3)
bert_model.to(DEVICE)
bert_model.eval()

print("Model loaded. Starting evaluation...")

bert_results = {}
for split in ["test_matched", "test_mismatched"]:
    result = evaluate_and_collect_wrongs_bert(
        model=bert_model,
        tokenizer=bert_tokenizer,
        split_name=split,
        output_dir="bert-base-turkish-cased-allnli_tr"
    )
    bert_results[split] = result

# Save metrics
with open("bert-base-turkish-cased-allnli_tr/metrics.json", "w") as f:
    json.dump({"trglue_mnli": bert_results}, f, indent=2)

# Clean up memory
del bert_model
del bert_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nBERT evaluation complete!")

Loading BERT-base-turkish-cased-allnli_tr...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2984.19it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: emrecan/bert-base-turkish-cased-allnli_tr
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Starting evaluation...

Evaluating on trglue_mnli / test_matched...


Inference: 100%|██████████| 282/282 [00:24<00:00, 11.60it/s]


  Accuracy: 0.7501, F1 Macro: 0.7434
  F1 per class: ['entailment: 0.8309', 'neutral: 0.6544', 'contradiction: 0.7450']
  Confusion matrix:
[[2678  167   79]
 [ 432 1666 1040]
 [ 412  121 2413]]
  Saved 2251 wrong predictions to bert-base-turkish-cased-allnli_tr/wrong_trglue_mnli_test_matched.csv

Evaluating on trglue_mnli / test_mismatched...


Inference: 100%|██████████| 289/289 [00:24<00:00, 11.93it/s]


  Accuracy: 0.8134, F1 Macro: 0.8068
  F1 per class: ['entailment: 0.8885', 'neutral: 0.7194', 'contradiction: 0.8126']
  Confusion matrix:
[[2877  148   76]
 [ 290 1841  912]
 [ 208   86 2779]]
  Saved 1720 wrong predictions to bert-base-turkish-cased-allnli_tr/wrong_trglue_mnli_test_mismatched.csv

BERT evaluation complete!


## Evaluate mDeBERTa-v3-base-mnli-xnli

In [14]:
print("=" * 80)
print("Loading mDeBERTa-v3-base-mnli-xnli...")
print("=" * 80)

mdeberta_model_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
mdeberta_tokenizer = AutoTokenizer.from_pretrained(mdeberta_model_name)
mdeberta_model = AutoModelForSequenceClassification.from_pretrained(mdeberta_model_name, ignore_mismatched_sizes=True)
mdeberta_model.to(DEVICE)
mdeberta_model.eval()

print("Model loaded. Starting evaluation...")

mdeberta_results = {}
for split in ["test_matched", "test_mismatched"]:
    result = evaluate_and_collect_wrongs_mdeberta(
        model=mdeberta_model,
        tokenizer=mdeberta_tokenizer,
        split_name=split,
        output_dir="mDeBERTa-v3-base-mnli-xnli"
    )
    mdeberta_results[split] = result

# Save metrics
with open("mDeBERTa-v3-base-mnli-xnli/metrics.json", "w") as f:
    json.dump({"trglue_mnli": mdeberta_results}, f, indent=2)

# Clean up memory
del mdeberta_model
del mdeberta_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nmDeBERTa evaluation complete!")

Loading mDeBERTa-v3-base-mnli-xnli...


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2351.82it/s, Materializing param=pooler.dense.weight]                                       
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded. Starting evaluation...

Evaluating on trglue_mnli / test_matched...


Inference:   0%|          | 0/282 [00:00<?, ?it/s]


KeyError: 'label'

## Evaluate Qwen2-7B-Instruct

In [ ]:
print("=" * 80)
print("Loading Qwen2-7B-Instruct...")
print("=" * 80)

from huggingface_hub import login
login()  # Required for gated models

qwen_model_name = "Qwen/Qwen2-7B-Instruct"
device_str = "mps" if torch.backends.mps.is_available() else 0 if torch.cuda.is_available() else -1

qwen_generator = pipeline(
    "text-generation",
    model=qwen_model_name,
    device=device_str,
    torch_dtype=torch.bfloat16 if torch.backends.mps.is_available() or torch.cuda.is_available() else torch.float32,
    max_length=None,
    trust_remote_code=True,
)

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

print("Model loaded. Starting evaluation...")

qwen_results = {}
for split in ["test_matched", "test_mismatched"]:
    result = evaluate_and_collect_wrongs_qwen(
        generator=qwen_generator,
        tokenizer=qwen_tokenizer,
        split_name=split,
        output_dir="Qwen2-7B-Instruct"
    )
    qwen_results[split] = result

# Save metrics
with open("Qwen2-7B-Instruct/metrics.json", "w") as f:
    json.dump({"trglue_mnli": qwen_results}, f, indent=2)

# Clean up memory
del qwen_generator
del qwen_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nQwen2-7B evaluation complete!")

## Summary

Compare the results of all three models on TrGLUE:

In [ ]:
# Create comparison table
comparison_data = []

for split in ["test_matched", "test_mismatched"]:
    comparison_data.append({
        "Split": split,
        "BERT_Accuracy": bert_results[split]["accuracy"],
        "BERT_F1_Macro": bert_results[split]["f1_macro"],
        "mDeBERTa_Accuracy": mdeberta_results[split]["accuracy"],
        "mDeBERTa_F1_Macro": mdeberta_results[split]["f1_macro"],
        "Qwen2-7B_Accuracy": qwen_results[split]["accuracy"],
        "Qwen2-7B_F1_Macro": qwen_results[split]["f1_macro"]
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison on TrGLUE MNLI:")
print(comparison_df)

# Save comparison
comparison_df.to_csv("trglue_model_comparison.csv", index=False)
print("\nSaved comparison to trglue_model_comparison.csv")